# LLM Data Prep — One-Time Tokenization & Google Drive Upload

Run this notebook **once** to:
1. Train (or load) the BPE tokenizer.
2. Tokenize the full corpus into `uint16` binary shards (`train_0000.bin`, …, `val.bin`).
3. Upload the tokenizer and all shards to Google Drive.

After that, `LLM_proto.ipynb` (and other training notebooks) can simply download the
pre-tokenized shards from Drive at the start of each session instead of re-running
the expensive tokenization step.

**Setup:**
- Set `GDRIVE_BINS_FOLDER_ID` to your Drive folder name/ID (e.g. `"LLM-data"`).
- Set the same value in `LLM_proto.ipynb` Cell 2 → `GDRIVE_BINS_FOLDER_ID`.
- On **Colab**: leave `GDRIVE_CREDENTIALS = ""` — OAuth popup handles auth.
- On **vast.ai / local**: set `GDRIVE_CREDENTIALS` to a service-account JSON path.

In [ ]:
# ── Cell 1: Install Dependencies ──
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("tokenizers", "datasets", "huggingface_hub")

# Google Drive API (needed outside Colab)
_pip("google-api-python-client", "google-auth-httplib2", "google-auth-oauthlib")

print("Dependencies installed.")

## 2. Settings
Edit the values below, then run all cells in order.

In [ ]:
# ── Cell 2: Settings ──
import os, sys, platform

# Add repo root to path so src/ imports work in Colab
REPO_ROOT = os.path.abspath(os.path.dirname(os.path.abspath('')) if '__file__' not in dir() else os.path.dirname(__file__))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ─── Paths ───────────────────────────────────────────────────────────────────
TOKENIZER_PATH = "tokenizer_data"      # local dir to save/load tokenizer.json
DATA_DIR       = "data"                # local dir for .bin shard output
DATA_CONFIG    = "configs/data.yaml"   # data.yaml with sources + processing cfg

# ─── Data sources ────────────────────────────────────────────────────────────
USE_HUGGINGFACE = False  # True = include HuggingFace datasets (slow on first run)
                         # False = only local data/custom/txt files

# ─── Google Drive ────────────────────────────────────────────────────────────
ENABLE_GDRIVE              = True
GDRIVE_TOKENIZER_FOLDER_ID = "LLM-data"  # Drive folder for tokenizer.json
GDRIVE_BINS_FOLDER_ID      = "LLM-data"  # Drive folder for .bin shards
                                          # (set this same value in LLM_proto.ipynb)
GDRIVE_CREDENTIALS         = ""           # service-account JSON (local/vast.ai only)
                                          # leave empty on Colab (uses OAuth)

# ─────────────────────────────────────────────────────────────────────────────
print(f"Repo root:  {REPO_ROOT}")
print(f"Tokenizer:  {TOKENIZER_PATH}")
print(f"Data dir:   {DATA_DIR}")
print(f"Config:     {DATA_CONFIG}")
print(f"HuggingFace sources: {'ENABLED' if USE_HUGGINGFACE else 'DISABLED'}")
print(f"GDrive bins folder:  {GDRIVE_BINS_FOLDER_ID!r}")

## 3. Connect to Google Drive
Mounts Drive on Colab, or verifies API credentials on other environments.

In [ ]:
# ── Cell 3: Connect to Google Drive ──
if ENABLE_GDRIVE:
    is_colab = "google.colab" in sys.modules

    if is_colab:
        from google.colab import drive
        MOUNT_POINT = "/content/drive"
        if not os.path.ismount(MOUNT_POINT):
            drive.mount(MOUNT_POINT)
            print(f"Google Drive mounted at {MOUNT_POINT}")
        else:
            print(f"Google Drive already mounted at {MOUNT_POINT}")

        # Show bins folder status
        if GDRIVE_BINS_FOLDER_ID:
            bins_path = os.path.join(MOUNT_POINT, "MyDrive", GDRIVE_BINS_FOLDER_ID)
            os.makedirs(bins_path, exist_ok=True)
            n_bins = len([f for f in os.listdir(bins_path) if f.endswith(".bin")])
            print(f"Bins folder: {bins_path} ({n_bins} existing .bin files)")
    else:
        if GDRIVE_CREDENTIALS and os.path.isfile(GDRIVE_CREDENTIALS):
            print(f"Drive credentials: {GDRIVE_CREDENTIALS}")
        elif GDRIVE_CREDENTIALS:
            print(f"Warning: credentials file not found: {GDRIVE_CREDENTIALS}")
        else:
            print("Drive API mode (Application Default Credentials)")

    print("Google Drive integration: ENABLED")
else:
    print("Google Drive integration: DISABLED")

## 4. Load or Train Tokenizer

Priority: **(1)** local `tokenizer_data/tokenizer.json` → **(2)** Google Drive → **(3)** train from scratch.  
After training, the tokenizer is automatically uploaded to Drive.

In [ ]:
# ── Cell 4: Load or Train Tokenizer ──
import yaml
from src.tokenizer import LLMTokenizer
from src.data import iter_texts_from_sources

tok_file = os.path.join(TOKENIZER_PATH, "tokenizer.json")
tok = None

# 1. Try local
if os.path.exists(tok_file):
    tok = LLMTokenizer(TOKENIZER_PATH)
    print(f"Tokenizer loaded from local path (vocab_size={tok.vocab_size})")

# 2. Try Google Drive
if tok is None and ENABLE_GDRIVE and GDRIVE_TOKENIZER_FOLDER_ID:
    print("Tokenizer not found locally, checking Google Drive...")
    try:
        from src.gdrive import download_from_gdrive
        download_from_gdrive(
            "tokenizer.json", GDRIVE_TOKENIZER_FOLDER_ID,
            TOKENIZER_PATH, credentials_path=GDRIVE_CREDENTIALS,
        )
        tok = LLMTokenizer(TOKENIZER_PATH)
        print(f"  Downloaded from Google Drive (vocab_size={tok.vocab_size})")
    except FileNotFoundError:
        print("  Not found on Google Drive either.")
    except Exception as e:
        print(f"  Google Drive download failed: {e}")

# 3. Train from scratch
if tok is None:
    print("Training tokenizer from scratch...")
    with open(DATA_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)

    tok_cfg = cfg["tokenizer"]
    sources = cfg.get("sources", [])
    if not USE_HUGGINGFACE:
        sources = [s for s in sources if s.get("type") != "huggingface"]
        print(f"  HuggingFace sources excluded. Active sources: {len(sources)}")
    num_samples = tok_cfg.get("num_samples", 50_000)
    vocab_size  = tok_cfg.get("vocab_size", 32_000)

    def text_iterator():
        count = 0
        for text in iter_texts_from_sources(sources):
            if count >= num_samples:
                break
            if text and len(text) > 50:
                yield text
                count += 1
        print(f"  Used {count:,} text samples for tokenizer training")

    tok = LLMTokenizer.train(
        texts=text_iterator(),
        vocab_size=vocab_size,
        save_path=TOKENIZER_PATH,
    )
    print(f"  Trained tokenizer (vocab_size={tok.vocab_size})")

    # Upload to Google Drive
    if ENABLE_GDRIVE and GDRIVE_TOKENIZER_FOLDER_ID:
        from src.gdrive import upload_to_gdrive
        upload_to_gdrive(
            tok_file, GDRIVE_TOKENIZER_FOLDER_ID,
            credentials_path=GDRIVE_CREDENTIALS,
        )
        print("  Uploaded tokenizer to Google Drive")

# Quick test
test_text = "Hello, world! This is a test of the tokenizer."
ids     = tok.encode(test_text, add_bos=True, add_eos=True)
decoded = tok.decode(ids)
print(f"\nTokenizer ready (vocab_size={tok.vocab_size})")
print(f"  Input:   {test_text}")
print(f"  Tokens:  {ids[:20]}{'...' if len(ids) > 20 else ''}")
print(f"  Decoded: {decoded}")

## 5. Tokenize Corpus → Binary Shards

Reads all configured text sources and writes `train_NNNN.bin` / `val.bin` shards to
`DATA_DIR`.  This is the slow step — typically 10-30 minutes on CPU.

> Re-running this cell is safe: existing `.bin` files are **not** overwritten.

In [ ]:
# ── Cell 5: Tokenize Data → Binary Shards ──
import copy, importlib, src.data as _data_mod
importlib.reload(_data_mod)

# Patch _iter_text_dir to read ALL files (not just *.txt)
def _iter_text_dir_all(source):
    """Read all non-hidden files in a directory, with or without extension."""
    path = source["path"]
    if not os.path.isdir(path):
        print(f"  Warning: text_dir path does not exist: {path}")
        print(f"    cwd: {os.getcwd()}, abs: {os.path.abspath(path)}")
        return
    all_files = sorted(
        os.path.join(root, fname)
        for root, _, files in os.walk(path)
        for fname in files
        if not fname.startswith(".")
    )
    print(f"  Found {len(all_files)} files in {path}")
    for fpath in all_files:
        try:
            text = open(fpath, "r", encoding="utf-8", errors="replace").read().strip()
        except (IOError, OSError):
            continue
        if text and len(text) >= 50:
            yield text

_data_mod._SOURCE_ITERATORS["text_dir"] = _iter_text_dir_all

from src.data import tokenize_and_save, load_data_config

_cfg     = load_data_config(DATA_CONFIG)
_sources = copy.deepcopy(_cfg.get("sources", []))
if not USE_HUGGINGFACE:
    _sources = [s for s in _sources if s.get("type") != "huggingface"]
    print(f"HuggingFace sources excluded. Active sources: {len(_sources)}")

# Resolve relative paths to absolute before any cwd changes
_cwd = os.getcwd()
for _src in _sources:
    if _src.get("type") in ("text_dir", "jsonl") and not os.path.isabs(_src.get("path", "")):
        _src["path"] = os.path.join(_cwd, _src["path"])

_proc       = _cfg.get("processing", {})
_output_dir = _proc.get("output_dir", DATA_DIR)
if not os.path.isabs(_output_dir):
    _output_dir = os.path.join(_cwd, _output_dir)

os.makedirs(_output_dir, exist_ok=True)

existing_bins = [f for f in os.listdir(_output_dir) if f.endswith(".bin")]
if existing_bins:
    print(f"Found {len(existing_bins)} existing .bin file(s) in {_output_dir}:")
    for _b in sorted(existing_bins):
        size_mb = os.path.getsize(os.path.join(_output_dir, _b)) / 1e6
        print(f"  {_b}  ({size_mb:.1f} MB)")
    print("Skipping tokenization for existing shards.")
    print("Delete .bin files and re-run to regenerate.")
else:
    print(f"Tokenizing data → {_output_dir}")
    tokenize_and_save(
        tokenizer_path=TOKENIZER_PATH,
        output_dir=_output_dir,
        sources=_sources,
        max_tokens=_proc.get("max_tokens"),
        shard_size=_proc.get("shard_size", 100_000_000),
    )
    print("Tokenization complete!")
    existing_bins = [f for f in os.listdir(_output_dir) if f.endswith(".bin")]
    for _b in sorted(existing_bins):
        size_mb = os.path.getsize(os.path.join(_output_dir, _b)) / 1e6
        print(f"  {_b}  ({size_mb:.1f} MB)")

## 6. Upload Shards to Google Drive

Uploads every `.bin` file from `DATA_DIR` to `GDRIVE_BINS_FOLDER_ID`.
Already-uploaded files are **overwritten** (upload is idempotent).  
Large shards use resumable uploads, so interrupted transfers can be safely re-run.

In [ ]:
# ── Cell 6: Upload .bin Shards to Google Drive ──
if not ENABLE_GDRIVE:
    print("GDrive disabled — skipping upload.")
elif not GDRIVE_BINS_FOLDER_ID:
    print("GDRIVE_BINS_FOLDER_ID is empty — set it in Cell 2 to enable upload.")
else:
    from src.gdrive import upload_dir_to_gdrive

    bins = sorted(f for f in os.listdir(_output_dir) if f.endswith(".bin"))
    print(f"Uploading {len(bins)} .bin shard(s) from {_output_dir}")
    print(f"  → Drive folder: {GDRIVE_BINS_FOLDER_ID}")
    print()

    results = upload_dir_to_gdrive(
        local_dir=_output_dir,
        folder_id=GDRIVE_BINS_FOLDER_ID,
        credentials_path=GDRIVE_CREDENTIALS,
    )

    print(f"\nUpload complete: {len(results)} file(s) uploaded.")
    print()
    print("Next steps:")
    print(f"  In LLM_proto.ipynb Cell 2, set:")
    print(f'    GDRIVE_BINS_FOLDER_ID = "{GDRIVE_BINS_FOLDER_ID}"')
    print("  Cell 7 will download the shards automatically on the next run.")